In [1]:
import pandas as pd 
#from google.colab import drive
from gensim.parsing.preprocessing import remove_stopwords
import re
from bertopic import BERTopic

docs = pd.read_excel("actions_count.xlsx")
docs.head()

/opt/anaconda3/envs/bertopic310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,tag_action,example,verb,object,n
0,write a poem,write a poem; write poetry; write poems; write...,write,poem,887
1,write code,write the code; write code; write some code; e...,write,code,775
2,write an essay,write an essay; write essays; just wrote an es...,write,essay,617
3,write a song,write a song; write the lyrics; write songs; w...,write,song,509
4,answer questions,answer questions; just answered a question; an...,answer,question,477


In [1]:
import matplotlib.pyplot as plt
from umap import UMAP
%matplotlib inline

docs["clean_text"] = docs["tag_action"].str.lower()

umap_model = UMAP(n_neighbors=15, n_components=5, 
                  min_dist=0.0, metric='cosine', random_state=1)

topic_model = BERTopic(embedding_model="stsb-mpnet-base-v2", umap_model=umap_model)

topics, _ = topic_model.fit_transform(docs["clean_text"])

freq_topics = topic_model.get_topic_info() 
print(freq_topics)

# Then, we extract the embeddings for each document
embeddings = topic_model._extract_embeddings(docs["clean_text"], method="document")

# Reducing dimensionality to 2d -> Play around with n_neighbors and min_dist
# Note that these 2D embeddings are very sensitive to hyperparameters
umap_embeddings = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', random_state=1).fit_transform(embeddings)

# Combine data
df = pd.DataFrame(umap_embeddings, columns=["x", "y"])
df["topic"] = topics

# Visualize topics
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(df['x'], df['y'], c=df['topic'], s=1, alpha=.3, cmap="Set1")
plt.show()